In [ ]:
using Distributed
using JLD2
using Plots

num_cores = length(Sys.cpu_info())
if nprocs()==1
    addprocs(num_cores; exeflags=`--project=$(Base.active_project())`)
end ;
@everywhere begin
    using LatticeAlgorithms
    using LinearAlgebra
    using Dates
end

In [ ]:
println("num_cores = $(num_cores)")

In [7]:
num_super_samples = 10
num_samples = Int(1e5)
Kmax = 100
Nv = 3

dmin, dmax = 5, 5
drange = dmin : 2 : dmax

# σrange = [0.595, 0.596, 0.597, 0.598, 0.599, 0.600, 0.601, 0.602, 0.603, 0.604, 0.605, 0.606, 0.607]
σrange = [0.606, 0.607]

σdrange = []
for σ in σrange
    for d in drange
        push!(σdrange, [σ, d])
    end
end
println(length(σdrange))

num_samples_each_core = Int(ceil(num_samples/num_cores))
num_samples = Int(num_samples_each_core * num_cores);
num_total_samples = num_super_samples * num_samples
println([num_samples_each_core, num_samples, num_total_samples])



logfile = "surface_square_mwms_$(dmin)_$(dmax)_$(min(σrange...))_$(max(σrange...))_$(Kmax)_$(Nv)_$(num_total_samples)_log.txt"
    open(logfile, "w") do file
end

In [ ]:
p_mld_list2 = Dict(σdrange.=>[0.0 for _ in 1 : length(σdrange)])
t_mld_list2 = Dict(σdrange.=>[0.0 for _ in 1 : length(σdrange)])

total_t = @elapsed for ind in 1 : num_super_samples  
    @time results = pmap(1:num_cores) do _        
        Ms = Dict()
        Ωs = Dict()
        Mperps = Dict()
        invtransposeMqs = Dict()
        for (ind_σd, σd) in enumerate(σdrange)
            σ, d = σd[1], Int(σd[2])
            M = surface_code_M(d) ; 
            Mperp = GKP_logical_operator_generator(M) 
            Ω = Ω_matrix(M)         
            invtransposeMq = inv(transpose(M))[1:2:end, 1:2:end]    
            Ms[d] = M
            Mperps[d] = Mperp
            Ωs[d] = Ω
            invtransposeMqs[d] = invtransposeMq
        end        

        p_mld_list = Dict(σdrange.=>[[] for _ in 1 : length(σdrange)])
        t_mld_list = Dict(σdrange.=>[0.0 for _ in 1 : length(σdrange)])        
        for (ind_σd, σd) in enumerate(σdrange)
            σ, d = σd[1], Int(σd[2])
            surface_code_z_logicals = surface_code_Z_logicals(d)
            surface_code_x_logicals = surface_code_X_logicals(d)
            
            p_mld = zeros(Kmax+1)
            t_mld = 0       

            σdtime = @elapsed for _ in 1 : num_samples_each_core
                ξ = σ * randn(2d^2)
                ξ2 = √(2π) * Ms[d] * inv(Ωs[d]) * ξ
                s = ξ2 - floor.(ξ2/(2π)) * 2π
                
                t_mld += @elapsed begin
                    ηs = -transpose(Ωs[d]*Mperps[d]) * s/√(2π) ;                     
                    ηsq = ηs[1:2:end]

                    ps_I, ps_X = LatticeAlgorithms.mwms_surface_square(ηsq, σ, Kmax; subspace="z", Nv=Nv)

                    ξq = ξ[1:2:end]                    
                    for k in 1 : Kmax+1
                        lstar = zeros(d^2)
                        p_I, p_X = ps_I[k], ps_X[k]
                        p_I > p_X ? (lstar) : (lstar[surface_code_x_logicals[1]] .= 1/√2 * √(2π))
                        recq = -ηsq + lstar
                        neterrorq = invtransposeMqs[d] * (recq+ξq) / √(2π)                
                        norm(round.(Int, neterrorq) - neterrorq) < 1e-10 ? nx = 0 : nx = 1
                        mod(nx, 2) == 0 ? (p_mld[k] += 1) : (p_mld[k] += 0)                            
                    end
                end  
            end
            p_mld_list[[σ, d]] = p_mld
            t_mld_list[[σ, d]] += t_mld
            
            if myid() == 2
                # Print the progress of the 2nd worker
                println(["$(ind)/$(num_super_samples), $(ind_σd)/$(length(σdrange)), $d, $(σdtime), $(string(now()))"])
                open(logfile, "a") do file
                    write(file, "$(ind)/$(num_super_samples), $(ind_σd)/$(length(σdrange)), $d, $(σdtime), $(string(now()))\n")
                end
            end
        end
        return p_mld_list, t_mld_list
    end ;     
    p_mld_list  = merge(.+, [res[1] for res in results]...)
    t_mld_list  = merge(+, [res[2] for res in results]...)
    
    p_mld_list2 = merge(.+, p_mld_list2, p_mld_list)
    t_mld_list2 = merge(.+, t_mld_list2, t_mld_list)
    
end

println(total_t)
map!(x->x/num_total_samples, values(p_mld_list2))
map!(x->x/num_total_samples, values(t_mld_list2))

fn = "surface_square_mwms_$(dmin)_$(dmax)_$(min(σrange...))_$(max(σrange...))_$(Kmax)_$(Nv)_$(num_total_samples).jld2"
jldsave(fn; 
    σrange=σrange, 
    drange=drange, 
    num_samples=num_total_samples,
    p_list = p_mld_list2,
    t_list = t_mld_list2,        
)    


In [ ]:
sort(load(fn))

# Check with existing data

In [ ]:
function get_p0list_sorted(p_list, drange, σrange)
    p0list_sorted = sort(p_list)
    p0list_sorted = collect(values(p0list_sorted))
    p0list_sorted = reshape(p0list_sorted, (length(drange), length(σrange)))
    p0list_sorted = [p0list_sorted[:,i] for i in 1:size(p0list_sorted,2)]
    return p0list_sorted
end

In [ ]:
new_data = sort(load(fn))
new_p_list = new_data["p_list"]
new_p_list_sorted = get_p0list_sorted(new_p_list, drange, σrange)

In [ ]:
old_data = sort(load("data/surface_square_mwms_5_15_0.596_0.607_400_3_1023680.jld2"))
old_p_list = Dict()
for (k, v) in old_data["p_list"]
    if k[2] ∈ drange && k[1] ∈ σrange
        old_p_list[k] = v
    end
end

old_p_list_sorted = get_p0list_sorted(old_p_list, drange, σrange)

In [ ]:
linecolors = get_color_palette(:auto, plot_color(:white))
linecolorind = 0    

g = plot()
for (ind_d, d) in enumerate(drange)
    for (ind_σ, σ) in enumerate(σrange)
        linecolorind +=1
        new_ps = new_p_list_sorted[ind_σ][ind_d]
        old_ps = old_p_list_sorted[ind_σ][ind_d]
        yerrnew = sqrt.(new_ps .* (1 .- new_ps) ./ num_total_samples)
        yerrold = sqrt.(old_ps .* (1 .- old_ps) ./ num_total_samples)
        plot!(new_ps, label="new data, d=$d, σ=$σ", marker=:circle, color=linecolors[linecolorind], yerr=1.5yerrnew)
        plot!(old_ps, label="old data, d=$d, σ=$σ", marker=:star, color=linecolors[linecolorind], yerr=1.5yerrold)
    end
end
plot!(xlabel="K", ylabel="fidelity", size=(1200, 400))